# 전조 증상 파악과 센서별 가중치 조절 방법

## 1. 이상 진동 전조 증상 파악 (Sliding Window 기법)

시계열 데이터에서 고장이 발생하기 전의 '전조 증상'을 잡으려면, 특정 시점의 데이터만 보는 것이 아니라 과거 일정 시간 동안의 흐름(추세)을 묶어서 모델에 입력해야 합니다. 이를 슬라이딩 윈도우(Sliding Window) 기법이라고 합니다.

### __*💡 원리*__

```
예를 들어 1초 단위로 수집되는 [온도, 습도, 진동, 전류] 데이터가 있을 때, 윈도우 크기를 5초로 설정하면 하나의 데이터 포인트가 5*4=20 차원의 벡터가 됩니다.
오토인코더는 이 20차원의 흐름을 학습하므로, 평소와 다른 '진동의 점진적 증가'나 '불규칙한 파형' 같은 전조 증상을 기가 막히게 잡아내어 복원 오차를 발생시킵니다.
```

### 🛠️ 파이썬 구현 코드 조각

In [ ]:
def create_sequences(data, window_size=5):
    sequences = []
    for i in range(len(data) - window_size + 1):
        # window_size만큼의 데이터를 묶어서 1차원으로 펼칩니다 (Flatten)
        seq = data[i : i + window_size].flatten()
        sequences.append(seq)
    return np.array(sequences)

# 윈도우 크기를 5로 설정 (예: 5초간의 데이터 묶음)
window_size = 5

# 원래 데이터가 (1000, 4)였다면, 변환 후 (996, 20)의 고차원 데이터가 됩니다.
X_train_seq = create_sequences(X_train_scaled, window_size)
X_test_norm_seq = create_sequences(X_test_norm_scaled, window_size)
X_test_fault_seq = create_sequences(X_test_fault_scaled, window_size)

# 이 데이터를 그대로 오토인코더(입력층 노드 수 20개로 수정)에 넣고 학습시킵니다.

## 2. 센서 종류에 따른 가중치 조절 (Weighted MSE)

* 일반적인 오토인코더는 모든 센서(온도, 습도, 진동, 전류)의 복원 오차를 동일한 비중으로 다룹니다.
* 하지만 건조기 도메인 특성상 "진동 센서의 미세한 변화가 고장 진단에 가장 결정적이다"라고 판단된다면, 진동 센서의 오차에 더 큰 가중치를 부여해야 합니다.

### 💡 원리

```
오토인코더의 손실 함수(Loss Function)를 기본 MSE(평균제곱오차)에서 사용자가 정의한 가중치 적용 MSE(Weighted MSE)로 변경합니다.
```

### 🛠️ 파이썬 구현 코드 조각 (TensorFlow/Keras)


In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

# 센서별 가중치 설정 (순서: 온도, 습도, 진동, 전류)
# 진동(세 번째) 센서의 오차에 5배의 가중치를 부여합니다.
sensor_weights = tf.constant([1.0, 1.0, 5.0, 1.0], dtype=tf.float32)

# 가중치가 적용된 커스텀 손실 함수 정의
def weighted_mse(y_true, y_pred):
    # 각 센서별 제곱 오차 계산
    squared_difference = tf.square(y_true - y_pred)
    # 가중치 곱하기
    weighted_difference = squared_difference * sensor_weights
    # 전체 평균 반환
    return tf.reduce_mean(weighted_difference, axis=-1)

# 모델 컴파일 시 커스텀 손실 함수 지정
autoencoder.compile(optimizer='adam', loss=weighted_mse)


### 🎯 요약 및 결합 활용

실무에서 가장 강력한 성능을 내는 방법은 이 두 가지를 결합하는 것입니다.

* 슬라이딩 윈도우로 5초간의 센서 데이터 흐름을 묶어 전조 증상을 담아내고,
* 가중치 손실 함수를 통해 진동 센서 영역의 오차에 가중치를 부여하여 모델을 학습시킵니다.

이렇게 구성하면 단순한 임계치 초과 고장뿐만 아니라, "진동이 서서히 불규칙해지는 전조 증상"까지 오토인코더가 매우 민감하게 반응하여 조기에 경보를 울릴 수 있습니다.


## 3. 완성된 Keras 오토인코더 코드

앞선 두 가지 기법(슬라이딩 윈도우와 센서별 가중치 조절)을 모두 결합하여, 건조기의 진동 전조 증상을 민감하게 잡아내는 완성된 Keras 오토인코더 코드입니다.

이 코드는 5초 동안의 센서 흐름(Window size = 5)을 묶어서 학습하며, 손실 함수에서 진동 센서에 5배의 가중치를 부여하여 진동 이상에 더 민감하게 반응하도록 설계되었습니다.

-----

### 💻 전조 증상 파악 + 센서 가중치 결합 오토인코더 코드

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape

# 1. 가상의 건조기 시계열 데이터 생성
np.random.seed(42)

# [정상 데이터 1,500개]
n_normal = 1500
t_train = np.linspace(0, 30 * np.pi, n_normal)
train_temp = 50 + 20 * np.sin(t_train) + np.random.normal(0, 0.5, n_normal)
train_humid = 55 + 35 * np.cos(t_train) + np.random.normal(0, 0.5, n_normal)
train_vib = 0.3 + 0.1 * np.sin(t_train * 2) + np.random.normal(0, 0.005, n_normal)
train_curr = 8 + 3 * np.cos(t_train / 2) + np.random.normal(0, 0.05, n_normal)
X_train_normal = np.column_stack([train_temp, train_humid, train_vib, train_curr])

# [테스트용 전조 증상 및 고장 데이터 300개]
# 뒤로 갈수록 진동이 서서히 불규칙해지며(전조 증상), 결국 튀는(고장) 데이터 생성
n_test = 300
t_test = np.linspace(30 * np.pi, 36 * np.pi, n_test)
test_temp = 50 + 20 * np.sin(t_test) + np.random.normal(0, 0.5, n_test)
test_humid = 55 + 35 * np.cos(t_test) + np.random.normal(0, 0.5, n_test)
# 후반부 100개 데이터에 진동 전조 및 고장 추가
test_vib = 0.3 + 0.1 * np.sin(t_test * 2) + np.random.normal(0, 0.005, n_test)
test_vib[200:260] += np.random.normal(0, 0.04, 60) # 전조 증상 (불규칙 진동)
test_vib[260:] += 0.2 # 고장 발생
test_curr = 8 + 3 * np.cos(t_test / 2) + np.random.normal(0, 0.05, n_test)
X_test = np.column_stack([test_temp, test_humid, test_vib, test_curr])


# 2. 데이터 전처리 (Min-Max Scaling)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)


# 3. 슬라이딩 윈도우(Sliding Window) 생성 함수
def create_windows(data, window_size=5):
    windows = []
    for i in range(len(data) - window_size + 1):
        windows.append(data[i : i + window_size])
    return np.array(windows)

window_size = 5 # 5개 시점(예: 5초)의 데이터를 묶음
X_train_windows = create_windows(X_train_scaled, window_size)
X_test_windows = create_windows(X_test_scaled, window_size)

print(f"학습 데이터 형태: {X_train_windows.shape}") # (1496, 5, 4) -> (샘플 수, 윈도우 크기, 센서 수)


# 4. 센서별 가중치 적용 Custom Loss Function 정의
# 데이터 순서: [온도, 습도, 진동, 전류] -> 진동에 가중치 5.0 부여
weights = tf.constant([1.0, 1.0, 5.0, 1.0], dtype=tf.float32)

def weighted_mse(y_true, y_pred):
    # y_true, y_pred 형태: (None, 5, 4)
    squared_diff = tf.square(y_true - y_pred)
    # 마지막 축(센서 축)에 가중치 곱하기
    weighted_diff = squared_diff * weights
    # 전체 평균 계산
    return tf.reduce_mean(weighted_diff)


# 5. Keras Autoencoder 모델 구성 (입출력 형태: (5, 4))
inputs = Input(shape=(window_size, 4))

# 인코더 (Flatten 후 압축)
x = Flatten()(inputs)
x = Dense(10, activation='relu')(x)
encoded = Dense(3, activation='relu')(x) # 20차원 -> 3차원으로 압축

# 디코더 (다시 복원 후 원래 형태로 Reshape)
x = Dense(10, activation='relu')(encoded)
x = Dense(window_size * 4, activation='sigmoid')(x)
decoded = Reshape((window_size, 4))(x)

autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer='adam', loss=weighted_mse)


# 6. 모델 학습 (정상 데이터의 윈도우 묶음으로만 수행)
autoencoder.fit(
    X_train_windows, X_train_windows,
    epochs=60,
    batch_size=32,
    shuffle=True,
    verbose=0
)


# 7. 복원 오차 및 임계치(Threshold) 계산
# 정상 학습 데이터의 오차 계산
train_preds = autoencoder.predict(X_train_windows, verbose=0)
# 각 시계열 윈도우 묶음별로 가중치가 반영된 MSE를 구합니다.
mse_train = np.mean(np.square(X_train_windows - train_preds) * weights.numpy(), axis=(1, 2))

# 임계치 설정 (평균 + 3 * 표준편차)
threshold = np.mean(mse_train) + 3 * np.std(mse_train)

# 테스트 데이터 오차 계산
test_preds = autoencoder.predict(X_test_windows, verbose=0)
mse_test = np.mean(np.square(X_test_windows - test_preds) * weights.numpy(), axis=(1, 2))


# 8. 시각화
plt.figure(figsize=(12, 6))
plt.plot(mse_test, label='Test Data Reconstruction Error', color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.5f})')

# 전조 증상 및 고장 구간 표시
plt.axvspan(200, 260, color='yellow', alpha=0.3, label='Slightly Abnormal (Precursor)')
plt.axvspan(260, len(mse_test), color='red', alpha=0.2, label='Fault Occurred')

plt.title('Fault & Precursor Detection (Weighted MSE + Sliding Window Autoencoder)')
plt.xlabel('Sliding Window Index')
plt.ylabel('Weighted Reconstruction Error')
plt.legend()
plt.grid(True)
plt.show()


### 🔍 코드의 핵심 구조 및 동작 원리

* create_windows:
  * (N, 4) 형태의 2차원 데이터를 (N-4, 5, 4) 형태의 3차원 데이터로 변환합니다.
  * 인공지능이 "5초간의 센서 패턴"을 한 번에 인지할 수 있게 됩니다.

* Flatten과 Reshape:
  * 3차원 윈도우 데이터를 압축하기 위해 신경망 내부에서 1차원(Flatten)으로 쭉 편 뒤 가공하고,
  * 출력할 때는 다시 원래의 (5, 4) 형태(Reshape)로 복원하여 원본과 1:1로 비교합니다.

* 가중치 매트릭스 브로드캐스팅:
  * weights.numpy()를 에러 매트릭스에 곱해주어,
  * 진동 값이 어긋날 때 전체 손실(Loss) 및 복원 오차가 훨씬 더 민감하게 튀어 오르도록 만듭니다.

시각화된 차트에서 노란색 전조 구간부터 에러가 임계치(Threshold)를 뚫고 올라가는 모습을 확인하실 수 있습니다.

## 4. 데이터 로드 가이드와 윈도우 크기(Window Size) 산정법

### 4.1 CSV 데이터 로드 및 전처리 가이드

* 일반적으로 건조기 센서 데이터는 시간(Datetime) 축과 센서별 컬럼으로 구성된 CSV 파일 형태를 띱니다.
* 데이터 분석 라이브러리인 Pandas를 활용하여 시계열 인덱스를 지정하고 로드하는 표준 코드입니다.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 불러오기
# '시간' 컬럼명이 'Timestamp'라고 가정합니다. 이를 인덱스로 지정하고 날짜 포맷으로 파싱합니다.
df = pd.read_csv('dryer_sensor_data.csv', parse_dates=['Timestamp'], index_col='Timestamp') #

# 2. 고장 진단에 필요한 센서 컬럼만 추출
# 예시: 온도, 습도, 진동, 전류 컬럼만 선택
sensor_cols = ['Temperature', 'Humidity', 'Vibration', 'Current']
df_sensors = df[sensor_cols]

# 3. 결측치(NaN) 처리
# 센서 누락값이 있다면 이전 값으로 채우거나(ffill) 선형 보간(interpolate)합니다.
df_sensors = df_sensors.interpolate(method='linear').ffill().bfill()

# 4. 스케일링 (0~1 사이 압축)
scaler = MinMaxScaler()
# 추후 고장 탐지의 오염을 막기 위해 '정상'으로 확신하는 기간의 데이터만 fit하는 것이 좋습니다!
scaled_data = scaler.fit_transform(df_sensors) 


### 4.2 윈도우 크기(Window Size) 산정 방법

윈도우 크기란 오토인코더가 한 번에 묶어서 인식할 '시간의 길이'입니다. 이를 너무 작게 잡으면 전조 증상의 추세를 놓치고, 너무 크게 잡으면 연산량이 늘어나고 미세한 변화가 묻히게 됩니다. (Medium)

적절한 윈도우 크기를 정하는 3가지 실무 기준은 다음과 같습니다.
* ① 센서 수집 주기(Sampling Rate) 기준 (가장 중요)
  * 센서가 1초에 1번 데이터를 기록하고, 건조기 진동의 이상 징후가 보통 10초 동안 서서히 나타난다면 윈도우 크기는 10이 적당합니다.
  * 만약 센서가 0.1초에 1번 기록한다면 동일한 10초의 흐름을 잡기 위해 윈도우 크기를 100으로 늘려야 합니다.
* ② 시스템의 물리적 반응 시간 기준
  * 건조기 내부의 온도가 오르거나 세탁물 무게에 의해 모터 전류가 변하는 물리적 주기를 고려합니다.
  * 예를 들어 건조통이 1바퀴 회전하는 데 걸리는 시간이 3초라면, 최소한 3초 이상의 데이터를 묶어야 회전 중 발생하는 규칙적인 진동의 패턴을 모델이 '정상'으로 학습할 수 있습니다. 
* ③ 추천하는 실험적 접근법
  * 실무에서는 보통 아래와 같이 단계별로 크기를 늘려가며 복원 오차(Loss)가 가장 안정적으로 떨어지는 구간을 찾습니다.
    * 초기 추천값: 수집 주기 기준으로 5초 ~ 30초 사이의 데이터 개수
    * 튜닝 팁: 윈도우 크기를 늘렸을 때 오토인코더의 Train Loss가 급격히 증가한다면, 모델이 너무 긴 시간의 복잡한 물리 법칙을 외우지 못하고 있다는 뜻이므로 크기를 다시 줄여야 합니다.

### 4.3 실적용 예

센서 수집 주기가 1초이고 컬럼이 20개 이상인 다변량 데이터는 전형적인 대용량 산업용 시계열 데이터의 형태입니다.

이러한 조건에서 모델이 과부하에 걸리지 않으면서도 전조 증상을 가장 잘 잡아낼 수 있는 1) 윈도우 크기 산정 기준과 2) 20개 이상의 다차원 컬럼을 효율적으로 처리하는 가이드를 안내해 드립니다.

#### 4.3.1 1초 주기 데이터의 윈도우 크기(Window Size) 산정

데이터가 1초마다 쌓이므로 윈도우 크기는 곧 '몇 초간의 묶음'인지를 의미합니다. 데이터의 특성을 고려하여 아래 3가지 후보군 중 하나를 선택해 실험해 보시는 것을 강력히 추천합니다.

* 후보 ① : 5 ~ 10 (초)
  * 목적: 연산 속도 최적화 및 순간적인 진동 스파이크(충격) 탐지
  * 설명: 윈도우 크기가 작기 때문에 모델이 가볍게 돌아가며, 실시간 고장 알람을 울리기에 가장 유리합니다.

* 후보 ② : 30 (초) ⭐️ [가장 추천]
  * 목적: '이상 진동'의 전조 증상(추세의 변화) 파악
  * 설명: 건조기의 모터나 드럼 회전 주기를 여러 번 포괄할 수 있는 시간입니다. 30초 동안 서서히 진동의 파동이 불규칙해지는 현상을 오토인코더가 학습하기에 가장 이상적인 크기입니다.
* 후보 ③ : 60 (초) 이상
  * 목적: 건조 사이클(가열-건조-냉각) 전체의 흐름을 반영
  * 설명: 20개가 넘는 컬럼(변수)에 윈도우 크기까지 60 이상으로 커지면 입력 차원이 $60*20=1,200$ 차원이 넘어가 모델이 너무 무거워질 수 있습니다. 연산 자원이 풍부할 때만 시도하는 것이 좋습니다.

#### 4.3.2 20개 이상 다차원 컬럼 처리 꿀팁

변수가 20개가 넘어가면 단순히 모든 데이터를 모델에 밀어 넣는 것보다 물리적 의미에 따라 분류하여 가공하는 것이 성능 향상의 핵심입니다.

* ① 도메인 지식을 활용한 변수 그룹화 (Grouping)
20개의 센서를 성격에 따라 묶고, 이전 답변에서 안내해 드린 센서별 가중치를 그룹 단위로 부여합니다.
  * 그룹 A (진동/전류): 고장 및 전조 증상에 직결되는 핵심 센서 --> 가중치 높음 (예: 5.0)
  * 그룹 B (온도/습도): 서서히 변하는 환경 센서 --> 가중치 중간 (예: 1.0)
  * 그룹 C (기기 설정값/도어 개폐 등): 단순 상태를 나타내는 디지털 값 --> 가중치 낮음 (예: 0.1)

* ② 상관관계 분석을 통한 변수 다이어트 
20개 중 서로 완벽하게 똑같이 움직이는 중복 센서가 있을 확률이 높습니다.
  * 가령 내부 온도 센서가 3개인데 3개의 데이터가 거의 일치한다면, Pandas의 .corr() 함수를 이용해 상관계수가 0.98 이상인 변수는 하나만 남기고 제거하여 모델의 부하를 줄여주는 것이 좋습니다.

#### 4.3.3 💻 20개 컬럼을 위한 모델 레이어 구성 팁

입력되는 변수가 많으므로(예: 30초 윈도우 X 20개 컬럼 = 600차원), Keras 모델의 뉴런 숫자를 조금 더 넓혀주어야 정보 손실 없이 복원이 가능합니다.

In [ ]:
# 다차원 변수를 수용할 수 있도록 신경망의 크기를 조금 키워줍니다.
inputs = Input(shape=(30, 20)) # 30초 윈도우, 20개 센서

x = Flatten()(inputs) # 600차원으로 펴짐
x = Dense(128, activation='relu')(x) # 첫 번째 축소
x = Dense(32, activation='relu')(x)  # 두 번째 축소
encoded = Dense(5, activation='relu')(x) # 5차원 잠재 공간으로 압축

# 디코더 (다시 복원)
x = Dense(32, activation='relu')(encoded)
x = Dense(128, activation='relu')(x)
x = Dense(30 * 20, activation='sigmoid')(x)
decoded = Reshape((30, 20))(x)

## 5. 파이프 라인과 상관계수 분석

건조기에서 수집되는 1초 주기, 20개 이상의 복합 센서 데이터를 다루기 위한 상관계수 분석 코드와 이를 결합한 오토인코더 머신러닝 파이프라인을 제공합니다.

### 5.1 상관계수 분석 및 중복 센서 제거 코드

변수 간의 선형적 유사도를 측정하여, 상관관계가 지나치게 높은(예: 0.90 이상) 센서 중 하나만 남기고 나머지는 제거하는 전처리 코드입니다.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# [가정] df는 20개 이상의 컬럼을 가진 건조기 센서 데이터프레임입니다.
# 실습을 위해 20개의 가상 컬럼을 가진 데이터프레임을 생성합니다.
np.random.seed(42)
rows = 1000
data = {f'Sensor_{i}': np.random.rand(rows) for i in range(1, 21)}
df = pd.DataFrame(data)

# 임의로 Sensor_1과 95% 일치하는 Sensor_21(가짜 중복 데이터)을 만들어 추가해 봅니다.
df['Sensor_21'] = df['Sensor_1'] + np.random.normal(0, 0.05, rows)

# --------------------------------------------------
# [STEP 1] 상관계수 행렬 계산
# --------------------------------------------------
# .abs()를 취해 양의 상관관계와 음의 상관관계를 모두 절댓값으로 평가합니다.
corr_matrix = df.corr().abs()

# --------------------------------------------------
# [STEP 2] 시각화 (히트맵)
# --------------------------------------------------
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='Blues', cbar=True)
plt.title('Sensor Data Correlation Heatmap')
plt.show()

# --------------------------------------------------
# [STEP 3] 특정 기준(예: 0.90) 이상의 중복 센서 추출 및 제거
# --------------------------------------------------
threshold = 0.90

# 대칭 행렬이므로 상삼각 행렬(Upper triangle)만 추출하여 중복 계산을 방지합니다.
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 기준점을 넘는 컬럼 찾기
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

print(f"▶ 원본 센서 개수: {df.shape[1]}개")
print(f"▶ 삭제 대상 센서 (상관계수 > {threshold}): {to_drop}")

# 중복 센서 제거 후 데이터프레임
df_reduced = df.drop(columns=to_drop)
print(f"▶ 정제 후 센서 개수: {df_reduced.shape[1]}개")

### 5.2 전체 고장 진단 파이프라인 (30초 윈도우 + 오토인코더)

상관계수 분석으로 정제된 df_reduced 데이터를 기반으로 고장을 탐지하는 완성형 파이프라인입니다.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape

# --------------------------------------------------
# [STEP 4] 데이터 스케일링
# --------------------------------------------------
# 오토인코더의 Sigmoid 출력을 위해 0~1 사이로 변환합니다.
scaler = MinMaxScaler()
# *주의: 실무에서는 고장 데이터가 섞이지 않은 '순수 정상 데이터' 기간으로만 fit 하세요.
scaled_data = scaler.fit_transform(df_reduced)

# --------------------------------------------------
# [STEP 5] 30초 슬라이딩 윈도우 생성
# --------------------------------------------------
def create_windows(data, window_size=30):
    windows = []
    for i in range(len(data) - window_size + 1):
        windows.append(data[i : i + window_size])
    return np.array(windows)

window_size = 30 # 1초 주기이므로 30초 동안의 흐름을 묶음
X_windows = create_windows(scaled_data, window_size)

# 데이터 형태: (총 샘플 수, 30, 축소된 센서 개수)
print(f"▶ 모델 입력 데이터 형태: {X_windows.shape}")

# --------------------------------------------------
# [STEP 6] 다차원 오토인코더 모델 구축
# --------------------------------------------------
n_features = X_windows.shape[2] # 최종 선택된 센서의 개수 (예: 20개)

inputs = Input(shape=(window_size, n_features))

# 인코더: 입력된 (30, n_features)의 2차원 시계열을 1차원으로 펴서 압축합니다.
x = Flatten()(inputs)
x = Dense(128, activation='relu')(x)
x = Dense(32, activation='relu')(x)
encoded = Dense(5, activation='relu')(x) # 5차원 잠재 공간(Bottleneck)

# 디코더: 다시 원래의 (30, n_features) 형태로 복원합니다.
x = Dense(32, activation='relu')(encoded)
x = Dense(128, activation='relu')(x)
x = Dense(window_size * n_features, activation='sigmoid')(x)
decoded = Reshape((window_size, n_features))(x)

autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# --------------------------------------------------
# [STEP 7] 모델 학습 및 고장 탐지 임계치 설정
# --------------------------------------------------
# 정상 데이터로만 학습을 수행합니다.
autoencoder.fit(
    X_windows, X_windows,
    epochs=50,
    batch_size=32,
    shuffle=True,
    verbose=1
)

# 복원 오차 계산
predictions = autoencoder.predict(X_windows)
mse = np.mean(np.square(X_windows - predictions), axis=(1, 2))

# 임계치(Threshold) 설정: 정상 오차의 평균 + 3 * 표준편차
threshold = np.mean(mse) + 3 * np.std(mse)
print(f"▶ 정상 기준 임계치: {threshold:.5f}")

# mse가 threshold를 넘어가면 해당 윈도우 시점에 고장(또는 전조 증상)이 발생한 것으로 판단합니다.

#### 💡 실무 적용 시 핵심 프로세스 요약

* df.corr()로 상관계수를 구한 뒤, 0.9 이상의 강한 연관성을 가진 센서들은 과감히 드롭하여 모델의 피처 수를 다이어트합니다.
* 1초 간격 데이터의 특성을 살려 window_size=30으로 묶어 30초간의 장기 추세를 오토인코더가 학습할 수 있도록 가이드합니다.

### 5.3 인덱스 설정과 데이터 파싱 팁

가장 먼저 확인해야 할 점은 보유하신 CSV 파일에 날짜와 시간 정보가 어떤 형태(포맷)로 들어있는지입니다. [1]

가전제품이나 산업용 센서 데이터에서 가장 흔하게 쓰이는 3가지 시간 포맷에 맞춰, 인덱스를 설정하고 데이터를 파싱하는 맞춤형 가이드를 안내해 드립니다.

#### 5.3.1 가장 흔한 3가지 시간 포맷별 파싱 방법

보유하신 CSV 파일의 첫 번째 컬럼(시간 데이터)이 어떤 형태인지 확인하시고 해당 코드를 적용해 보세요.

_*① 일반적인 날짜와 시간 형태 (가장 추천)*_

* 데이터 예시: 2026-04-13 10:30:01 또는 2026/04/13 10:30:01
* 파싱 코드:

In [ ]:
import pandas as pd

# parse_dates에 시간 컬럼명을 적어주면 판다스가 알아서 datetime 객체로 변환합니다.
df = pd.read_csv('dryer_data.csv', parse_dates=['Timestamp'], index_col='Timestamp')
print(df.index) # 'DatetimeIndex'로 출력되어야 정상입니다.

_*② 유닉스 타임스탬프 (Unix Timestamp) 형태*_

* 데이터 예시: 1712994601 (1970년 1월 1일부터 경과한 초 단위 시간)
* 파싱 코드: 숫자 형태로 되어 있어 판다스가 날짜로 바로 인식하지 못하므로, 불러온 뒤 단위를 지정해 변환해야 합니다.

In [ ]:
df = pd.read_csv('dryer_data.csv')
# 초(s) 단위인 경우 unit='s'를 지정하여 변환합니다.
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Timestamp', inplace=True)

_*③ 단순히 1, 2, 3... 숫자로 기록된 경우 (상대 시간)*_

* 데이터 예시: 1, 2, 3 (1초 주기로 수집되었으나 실제 시각이 아닌 경과 초만 기록된 경우)
* 파싱 코드: 이 경우 억지로 날짜로 바꿀 필요 없이, 숫자 그대로 인덱스로 쓰셔도 무방합니다.

In [ ]:
# 'Time_Step' 컬럼을 그대로 인덱스로 지정
df = pd.read_csv('dryer_data.csv', index_col='Time_Step')

#### 5.3.2 1초 주기 데이터 검증 및 리샘플링 (실무 꿀팁)

센서가 1초 주기로 수집된다고 해도, 통신 지연이나 기기 오류로 인해 데이터가 2초 뒤에 찍히거나 누락되는 경우가 빈번하게 발생합니다. 오토인코더 모델에 규칙적인 30초 윈도우를 넣으려면 시간 축의 간격을 균일하게 만들어 주어야 합니다.

_*🛠️ 시간 간격 균일화 및 결측치 처리 코드*_

In [ ]:
# 1. 인덱스를 기준으로 시간 순서대로 정렬합니다.
df = df.sort_index()

# 2. 1초(1S) 간격으로 데이터를 재배열합니다. 
# 만약 누락된 초(S)가 있다면 빈 행(NaN)이 자동으로 생성됩니다.
df_resampled = df.resample('1S').mean()

# 3. 누락되어 발생한 결측치(NaN)를 앞뒤 데이터의 평균값으로 부드럽게 채웁니다.
df_final = df_resampled.interpolate(method='linear')

print(f"누락 처리 전 데이터 개수: {len(df)}")
print(f"1초 간격 정렬 후 데이터 개수: {len(df_final)}")